# Notebook 7: Real Benchmarks, Link Prediction & Capstone Project

This final notebook consolidates everything. We'll run models on the **Open Graph Benchmark (OGB)** — the gold standard for GNN evaluation — cover **link prediction** (a completely different task type), and end with a capstone project.

**Prerequisites:** All previous notebooks

**What you'll learn:**
- OGB: why it exists, how to use it
- Link prediction: the task, negative sampling, AUC evaluation
- Comparing GCN / GAT / GraphSAGE / GIN side by side
- GNN tricks that actually matter in practice
- Capstone: build a complete GNN recommendation system

**After this notebook:** you'll have a complete GNN toolkit and a full project to show.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
from torch_geometric.nn import GCNConv, GATConv, SAGEConv, GINConv
from torch_geometric.nn import global_add_pool, global_mean_pool
from torch_geometric.utils import negative_sampling, train_test_split_edges
from torch_geometric.datasets import Planetoid
from torch_geometric.loader import DataLoader
import warnings
warnings.filterwarnings('ignore')

torch.manual_seed(42)
np.random.seed(42)

print('All imports OK')

---
## 1. The Open Graph Benchmark (OGB)

OGB (Hu et al., 2020) was created because older benchmarks (Cora, MUTAG) are too small for meaningful comparisons. Key properties:
- **Realistic scale**: millions of nodes and edges
- **Standard splits**: no more dataset cherry-picking
- **Multiple tasks**: node, link, and graph prediction
- **Leaderboards**: compare against published methods

| Dataset | Task | Nodes | Edges | Metric |
|---------|------|-------|-------|--------|
| ogbn-arxiv | Node classification | 169K | 1.2M | Accuracy |
| ogbn-products | Node classification | 2.4M | 61M | Accuracy |
| ogbl-collab | Link prediction | 235K | 1.3M | Hits@50 |
| ogbg-molhiv | Graph classification | avg 25 | avg 27 | ROC-AUC |

We'll use **ogbn-arxiv** (manageable size) and **ogbg-molhiv** (molecular property prediction).

In [ ]:
# OGB datasets — install: pip install ogb
try:
    from ogb.nodeproppred import PygNodePropPredDataset, Evaluator as NodeEval
    from ogb.graphproppred import PygGraphPropPredDataset, Evaluator as GraphEval
    OGB_AVAILABLE = True
    print('OGB available!')
except ImportError:
    OGB_AVAILABLE = False
    print('OGB not installed. Run: pip install ogb')
    print('Falling back to Cora + MUTAG for demos.')

if OGB_AVAILABLE:
    # ogbn-arxiv: citation network of CS arXiv papers
    dataset_arxiv = PygNodePropPredDataset(name='ogbn-arxiv', root='/tmp/ogbn')
    data_arxiv = dataset_arxiv[0]
    split_idx = dataset_arxiv.get_idx_split()
    
    print(f'ogbn-arxiv:')
    print(f'  Nodes:    {data_arxiv.num_nodes:>8,}')
    print(f'  Edges:    {data_arxiv.num_edges:>8,}')
    print(f'  Features: {data_arxiv.num_node_features:>8}')
    print(f'  Classes:  {dataset_arxiv.num_classes:>8}')
    print(f'  Train:    {len(split_idx["train"]):>8,}')
    print(f'  Val:      {len(split_idx["valid"]):>8,}')
    print(f'  Test:     {len(split_idx["test"]):>8,}')

In [ ]:
# ============================================================
# GNN for ogbn-arxiv (large-scale node classification)
# ============================================================
if OGB_AVAILABLE:
    from torch_geometric.loader import NeighborLoader
    
    class ArxivGNN(nn.Module):
        """3-layer GraphSAGE for large-scale node classification."""
        def __init__(self, in_channels, hidden_channels, out_channels, num_layers=3, dropout=0.5):
            super().__init__()
            self.dropout = dropout
            self.convs = nn.ModuleList()
            self.bns = nn.ModuleList()
            
            self.convs.append(SAGEConv(in_channels, hidden_channels))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
            for _ in range(num_layers - 2):
                self.convs.append(SAGEConv(hidden_channels, hidden_channels))
                self.bns.append(nn.BatchNorm1d(hidden_channels))
            self.convs.append(SAGEConv(hidden_channels, out_channels))
        
        def forward(self, x, edge_index):
            for conv, bn in zip(self.convs[:-1], self.bns):
                x = conv(x, edge_index)
                x = bn(x)
                x = F.relu(x)
                x = F.dropout(x, p=self.dropout, training=self.training)
            return self.convs[-1](x, edge_index)
    
    # Create masks compatible with PyG NeighborLoader
    n = data_arxiv.num_nodes
    train_mask = torch.zeros(n, dtype=torch.bool)
    val_mask = torch.zeros(n, dtype=torch.bool)
    test_mask = torch.zeros(n, dtype=torch.bool)
    train_mask[split_idx['train']] = True
    val_mask[split_idx['valid']] = True
    test_mask[split_idx['test']] = True
    data_arxiv.train_mask = train_mask
    data_arxiv.val_mask = val_mask
    data_arxiv.test_mask = test_mask
    
    train_loader_arxiv = NeighborLoader(
        data_arxiv,
        num_neighbors=[15, 10, 5],  # 3 layers
        batch_size=1024,
        input_nodes=data_arxiv.train_mask,
        shuffle=True
    )
    
    model_arxiv = ArxivGNN(
        in_channels=data_arxiv.num_node_features,
        hidden_channels=256,
        out_channels=dataset_arxiv.num_classes
    )
    print(f'ArxivGNN params: {sum(p.numel() for p in model_arxiv.parameters()):,}')
    print('Model ready for training (skipping full training run — would take ~10 min)')
    print('To train: use the same train_minibatch / eval_minibatch from Notebook 5')
else:
    print('(OGB not available — skipping ogbn-arxiv demo)')

---
## 2. Link Prediction — A Different Task

**Task:** given a graph, predict which pairs of nodes *should* be connected (but aren't yet — or are hidden).

**Applications:**
- Recommender systems: will user A buy product B?
- Knowledge graph completion: is there a relation between entities?
- Drug discovery: will drug A bind to protein B?
- Social networks: should we suggest this friend connection?

### How it works:
1. **Training:** hide some edges → train GNN to reconstruct them
2. **Negative sampling:** create fake edges that don't exist → the model must distinguish real vs. fake
3. **Scoring:** use node embeddings — score(u, v) = dot(h_u, h_v) or MLP(concat(h_u, h_v))
4. **Metric:** AUC-ROC (area under the ROC curve)

In [ ]:
# ============================================================
# Link Prediction on Cora
# ============================================================
from torch_geometric.datasets import Planetoid
from torch_geometric.utils import train_test_split_edges
from sklearn.metrics import roc_auc_score

dataset_lp = Planetoid(root='/tmp/Cora', name='Cora')
data_lp = dataset_lp[0]

# Split edges into train/val/test
# This hides some edges and creates negative samples
data_lp = train_test_split_edges(data_lp)

print('Link prediction split:')
print(f'  Train pos edges: {data_lp.train_pos_edge_index.shape[1]}')
print(f'  Val pos edges:   {data_lp.val_pos_edge_index.shape[1]}')
print(f'  Val neg edges:   {data_lp.val_neg_edge_index.shape[1]}')
print(f'  Test pos edges:  {data_lp.test_pos_edge_index.shape[1]}')
print(f'  Test neg edges:  {data_lp.test_neg_edge_index.shape[1]}')

In [ ]:
class LinkPredGNN(nn.Module):
    """GNN encoder for link prediction. Outputs node embeddings."""
    
    def __init__(self, in_channels, hidden_channels, out_channels):
        super().__init__()
        self.conv1 = GCNConv(in_channels, hidden_channels)
        self.conv2 = GCNConv(hidden_channels, out_channels)
    
    def encode(self, x, edge_index):
        x = F.relu(self.conv1(x, edge_index))
        return self.conv2(x, edge_index)
    
    def decode(self, z, edge_index):
        """Score edges using dot product of node embeddings."""
        src, dst = edge_index
        return (z[src] * z[dst]).sum(dim=-1)  # dot product per edge
    
    def decode_all(self, z):
        """Score ALL possible edges — for small graphs only!"""
        return (z @ z.T)  # full similarity matrix


torch.manual_seed(42)
model_lp = LinkPredGNN(
    in_channels=data_lp.num_features,
    hidden_channels=64,
    out_channels=64  # embedding dimension
)
optimizer_lp = torch.optim.Adam(model_lp.parameters(), lr=0.01)
print(model_lp)

In [ ]:
def train_link_pred(model, data, optimizer):
    model.train()
    optimizer.zero_grad()
    
    # Encode all nodes
    z = model.encode(data.x, data.train_pos_edge_index)
    
    # Positive edges (real connections)
    pos_pred = model.decode(z, data.train_pos_edge_index)
    
    # Negative edges (random non-existent connections)
    neg_edge_index = negative_sampling(
        edge_index=data.train_pos_edge_index,
        num_nodes=data.num_nodes,
        num_neg_samples=data.train_pos_edge_index.shape[1]
    )
    neg_pred = model.decode(z, neg_edge_index)
    
    # Binary cross-entropy: pos=1, neg=0
    pos_labels = torch.ones(pos_pred.size(0))
    neg_labels = torch.zeros(neg_pred.size(0))
    
    loss = F.binary_cross_entropy_with_logits(
        torch.cat([pos_pred, neg_pred]),
        torch.cat([pos_labels, neg_labels])
    )
    loss.backward()
    optimizer.step()
    return loss.item()


@torch.no_grad()
def eval_link_pred(model, data, pos_edge_index, neg_edge_index):
    model.eval()
    z = model.encode(data.x, data.train_pos_edge_index)
    
    pos_pred = torch.sigmoid(model.decode(z, pos_edge_index)).cpu()
    neg_pred = torch.sigmoid(model.decode(z, neg_edge_index)).cpu()
    
    pred = torch.cat([pos_pred, neg_pred]).numpy()
    labels = torch.cat([torch.ones(pos_pred.size(0)),
                        torch.zeros(neg_pred.size(0))]).numpy()
    return roc_auc_score(labels, pred)


history_lp = {'loss': [], 'val_auc': [], 'test_auc': []}
best_val_auc = 0; best_test_auc = 0

for epoch in range(1, 201):
    loss = train_link_pred(model_lp, data_lp, optimizer_lp)
    val_auc = eval_link_pred(model_lp, data_lp,
                              data_lp.val_pos_edge_index,
                              data_lp.val_neg_edge_index)
    test_auc = eval_link_pred(model_lp, data_lp,
                               data_lp.test_pos_edge_index,
                               data_lp.test_neg_edge_index)
    
    history_lp['loss'].append(loss)
    history_lp['val_auc'].append(val_auc)
    history_lp['test_auc'].append(test_auc)
    
    if val_auc > best_val_auc:
        best_val_auc, best_test_auc = val_auc, test_auc
    
    if epoch % 50 == 0:
        print(f'Epoch {epoch:>3} | Loss: {loss:.4f} | Val AUC: {val_auc:.4f} | Test AUC: {test_auc:.4f}')

print(f'\nBest Val AUC: {best_val_auc:.4f} | Test AUC: {best_test_auc:.4f}')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].plot(history_lp['loss'], color='red')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('BCE Loss')
axes[0].set_title('Link Prediction Training Loss')

axes[1].plot(history_lp['val_auc'], label='Val AUC', color='orange')
axes[1].plot(history_lp['test_auc'], label='Test AUC', color='green', linestyle='--')
axes[1].axhline(best_test_auc, color='gray', linestyle=':', label=f'Best test: {best_test_auc:.4f}')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('AUC-ROC')
axes[1].set_title('Link Prediction AUC (Cora)')
axes[1].legend()

plt.tight_layout()
plt.show()

---
## 3. Model Comparison — All Architectures on Cora

In [ ]:
from torch_geometric.datasets import Planetoid

dataset = Planetoid(root='/tmp/Cora', name='Cora')
data = dataset[0]

def make_gcn(in_ch, hidden, out_ch):
    class GCN(nn.Module):
        def __init__(self):
            super().__init__()
            self.c1 = GCNConv(in_ch, hidden, cached=True)
            self.c2 = GCNConv(hidden, out_ch, cached=True)
        def forward(self, x, ei):
            return self.c2(F.dropout(F.relu(self.c1(x, ei)), 0.5, self.training), ei)
    return GCN()

def make_gat(in_ch, hidden, out_ch):
    class GAT(nn.Module):
        def __init__(self):
            super().__init__()
            self.c1 = GATConv(in_ch, hidden, heads=8, dropout=0.6)
            self.c2 = GATConv(hidden*8, out_ch, heads=1, concat=False, dropout=0.6)
        def forward(self, x, ei):
            x = F.dropout(x, 0.6, self.training)
            x = F.elu(self.c1(x, ei))
            x = F.dropout(x, 0.6, self.training)
            return self.c2(x, ei)
    return GAT()

def make_sage(in_ch, hidden, out_ch):
    class SAGE(nn.Module):
        def __init__(self):
            super().__init__()
            self.c1 = SAGEConv(in_ch, hidden)
            self.c2 = SAGEConv(hidden, out_ch)
        def forward(self, x, ei):
            return self.c2(F.dropout(F.relu(self.c1(x, ei)), 0.5, self.training), ei)
    return SAGE()


def train_and_eval(model, data, lr=0.01, wd=5e-4, epochs=300):
    opt = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=wd)
    best_val, best_test = 0, 0
    for epoch in range(epochs):
        model.train(); opt.zero_grad()
        out = model(data.x, data.edge_index)
        F.cross_entropy(out[data.train_mask], data.y[data.train_mask]).backward()
        opt.step()
        model.eval()
        with torch.no_grad():
            pred = model(data.x, data.edge_index).argmax(1)
        v = (pred[data.val_mask] == data.y[data.val_mask]).float().mean().item()
        t = (pred[data.test_mask] == data.y[data.test_mask]).float().mean().item()
        if v > best_val: best_val, best_test = v, t
    return best_val, best_test


IN = dataset.num_features
HID = 64
OUT = dataset.num_classes

models_to_compare = {
    'GCN': (make_gcn(IN, HID, OUT), 0.01, 5e-4),
    'GAT': (make_gat(IN, 8, OUT), 0.005, 5e-4),
    'GraphSAGE': (make_sage(IN, 256, OUT), 0.01, 0),
}

results = {}
for name, (m, lr, wd) in models_to_compare.items():
    torch.manual_seed(42)
    v, t = train_and_eval(m, data, lr=lr, wd=wd, epochs=300)
    results[name] = (v, t)
    print(f'{name:<12} | Val: {v:.4f} | Test: {t:.4f}')

In [ ]:
# Visualization of comparison
fig, ax = plt.subplots(figsize=(9, 5))

names = list(results.keys())
vals = [results[n][0] for n in names]
tests = [results[n][1] for n in names]

x = np.arange(len(names))
w = 0.35
bars1 = ax.bar(x - w/2, vals, w, label='Val Acc', color='steelblue', alpha=0.8)
bars2 = ax.bar(x + w/2, tests, w, label='Test Acc', color='coral', alpha=0.8)

for bars in [bars1, bars2]:
    for bar in bars:
        h = bar.get_height()
        ax.text(bar.get_x() + bar.get_width()/2, h + 0.003,
                f'{h:.3f}', ha='center', va='bottom', fontsize=9)

ax.set_ylim(0.78, 0.90)
ax.set_xticks(x)
ax.set_xticklabels(names, fontsize=12)
ax.set_ylabel('Accuracy')
ax.set_title('Cora Node Classification — Model Comparison')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. GNN Tricks That Actually Matter

After running all these experiments, here's a practical guide to what actually helps:

| Trick | When it helps | Notes |
|-------|--------------|-------|
| **Dropout on inputs** | Always | Especially on Cora-style sparse features |
| **Weight decay (L2)** | Small graphs | 5e-4 is a solid default |
| **BatchNorm** | Deep models | Stabilizes training |
| **Residual connections** | 3+ layers | Prevents over-smoothing |
| **Learning rate schedule** | Graph classification | StepLR or cosine annealing |
| **Neighbor sampling** | Large graphs | Essential for graphs >100K nodes |
| **Edge dropout** | Dense graphs | Acts as graph augmentation |
| **Feature normalization** | Always | Normalize node features before training |

In [ ]:
# Improved GCN with all tricks
class ImprovedGCN(nn.Module):
    def __init__(self, in_channels, hidden_channels, out_channels,
                 num_layers=3, dropout=0.5):
        super().__init__()
        self.dropout = dropout
        self.convs = nn.ModuleList()
        self.bns = nn.ModuleList()
        
        self.convs.append(GCNConv(in_channels, hidden_channels, cached=True))
        self.bns.append(nn.BatchNorm1d(hidden_channels))
        for _ in range(num_layers - 2):
            self.convs.append(GCNConv(hidden_channels, hidden_channels, cached=True))
            self.bns.append(nn.BatchNorm1d(hidden_channels))
        self.convs.append(GCNConv(hidden_channels, out_channels, cached=True))
    
    def forward(self, x, edge_index):
        # Input dropout
        x = F.dropout(x, p=self.dropout, training=self.training)
        
        for i, (conv, bn) in enumerate(zip(self.convs[:-1], self.bns)):
            x_new = conv(x, edge_index)
            x_new = bn(x_new)
            x_new = F.relu(x_new)
            # Residual connection (if dimensions match)
            if x.shape == x_new.shape:
                x_new = x_new + x
            x = F.dropout(x_new, p=self.dropout, training=self.training)
        
        return self.convs[-1](x, edge_index)


torch.manual_seed(42)
improved = ImprovedGCN(dataset.num_features, 64, dataset.num_classes, num_layers=3)
val_i, test_i = train_and_eval(improved, data, lr=0.01, wd=5e-4, epochs=300)
print(f'Improved GCN | Val: {val_i:.4f} | Test: {test_i:.4f}')
print(f'Basic GCN    | Val: {results["GCN"][0]:.4f} | Test: {results["GCN"][1]:.4f}')

---
# CAPSTONE PROJECT: Movie Recommendation with GNN

**Task:** Build a **bipartite GNN-based recommendation system** using user-movie interactions.

A recommendation system is naturally a *link prediction* problem on a bipartite graph:
- Nodes: users + movies (two types)
- Edges: user has rated movie (existing links)
- Task: predict which movies a user will rate next (link prediction)

**Dataset:** Use the MovieLens-100K dataset (download instructions in cell below).

**Requirements:**
1. Build a bipartite graph: users = one set of nodes, movies = another set
2. Use movie genre tags as node features for movie nodes; user degree as feature for user nodes
3. Hide 20% of edges for testing, 10% for validation
4. Build a 2-layer GNN encoder that produces embeddings for both users and movies
5. Score user-movie pairs with dot product
6. Train with negative sampling (for each positive user-movie pair, sample K negatives)
7. Evaluate with AUC-ROC and Hits@K metrics
8. Visualize: top-10 movie recommendations for 5 random users

**Extension ideas:**
- Try `HeteroData` (PyG's heterogeneous graph support) for proper user/movie separation
- Add movie year, user age as additional features
- Try different aggregations for user nodes vs. movie nodes

In [ ]:
# CAPSTONE SKELETON
# Download MovieLens: https://grouplens.org/datasets/movielens/100k/
# Or use this quick synthetic dataset for structure:

import torch
import numpy as np
from torch_geometric.data import Data
from torch_geometric.utils import negative_sampling

# --- Option A: Synthetic bipartite graph ---
# (Replace with real MovieLens data for the full project)

def create_synthetic_bipartite(num_users=100, num_movies=500, num_ratings=2000, seed=42):
    """Create a synthetic user-movie bipartite graph."""
    rng = np.random.RandomState(seed)
    
    # Users: node indices 0..num_users-1
    # Movies: node indices num_users..num_users+num_movies-1
    N = num_users + num_movies
    
    # Random ratings (edges)
    users = rng.randint(0, num_users, size=num_ratings)
    movies = rng.randint(num_users, N, size=num_ratings)
    
    # Deduplicate
    edge_pairs = list(set(zip(users.tolist(), movies.tolist())))
    src = [e[0] for e in edge_pairs]
    dst = [e[1] for e in edge_pairs]
    # Make undirected
    edge_index = torch.tensor(
        [src + dst, dst + src], dtype=torch.long
    )
    
    # Node features:
    # Users: [degree_normalized, random_feature]
    # Movies: [genre_1, genre_2, genre_3, genre_4, genre_5, random]
    user_feat = torch.randn(num_users, 2)
    movie_feat = torch.randint(0, 2, (num_movies, 6)).float()
    x = torch.cat([user_feat, movie_feat], dim=0)
    
    # Ratings as edge weights (simulate 1-5 stars)
    edge_attr = torch.randint(1, 6, (len(edge_pairs) * 2,), dtype=torch.float)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr), num_users, num_movies


data_rec, num_users, num_movies = create_synthetic_bipartite()
print(f'Recommendation graph:')
print(f'  Total nodes: {data_rec.num_nodes} ({num_users} users + {num_movies} movies)')
print(f'  Edges (ratings): {data_rec.num_edges}')
print(f'  Node features: {data_rec.x.shape}')

In [ ]:
# ============================================================
# TODO 1: GNN Encoder
# ============================================================
class RecommenderGNN(nn.Module):
    """
    GNN encoder for user-movie bipartite graph.
    Produces embeddings for all nodes (users + movies).
    """
    def __init__(self, in_channels, hidden_channels, out_channels, num_layers=2):
        super().__init__()
        # TODO: define GNN layers (SAGEConv works well for bipartite)
        # self.convs = nn.ModuleList(...)
        pass
    
    def forward(self, x, edge_index):
        # TODO: forward pass -> return node embeddings
        pass
    
    def predict(self, z, user_idx, movie_idx):
        """Score a batch of (user, movie) pairs."""
        # TODO: dot product of user and movie embeddings
        # return (z[user_idx] * z[movie_idx]).sum(dim=-1)
        pass


# ============================================================
# TODO 2: Train/val/test edge split
# ============================================================
# Split edges: 70% train, 10% val, 20% test
# For bipartite: make sure to split (user->movie) edges only
# Hint: use torch_geometric.utils.train_test_split_edges

# ============================================================
# TODO 3: Training with negative sampling
# ============================================================
def train_recommender(model, data, optimizer, train_pos_edges):
    model.train()
    optimizer.zero_grad()
    
    # TODO:
    # 1. Encode all nodes
    # 2. Score positive edges -> pos_pred
    # 3. Sample negative edges (negative_sampling)
    # 4. Score negative edges -> neg_pred
    # 5. Binary cross-entropy loss
    # 6. Backward + step
    pass


# ============================================================
# TODO 4: Evaluation (AUC and Hits@K)
# ============================================================
def hits_at_k(model, z, pos_edges, all_movie_indices, k=10):
    """
    For each user in pos_edges, rank ALL movies by score.
    Count how many true positives appear in top-K.
    """
    # TODO: for each unique user, compute scores against all movies,
    # take top-K, check if true positive is in top-K
    pass


# ============================================================
# TODO 5: Recommendation visualization
# ============================================================
def get_top_k_recommendations(model, z, user_idx, movie_indices, k=10):
    """
    Return top-K movie indices for a given user.
    """
    # TODO: score user against all movies, return top-K movie indices
    pass


print('Fill in the TODO sections to build your recommendation system!')
print('This is your capstone — take your time with it.')

In [ ]:
# OPTIONAL: Load real MovieLens data
# Uncomment and run if you've downloaded ml-100k from:
# https://grouplens.org/datasets/movielens/100k/

'''
import pandas as pd

def load_movielens(data_path: str):
    # Load ratings
    ratings = pd.read_csv(f'{data_path}/u.data', sep='\t',
                          names=['user_id', 'item_id', 'rating', 'timestamp'])
    
    # Load movie info
    movies = pd.read_csv(f'{data_path}/u.item', sep='|', encoding='latin-1',
                         names=['item_id', 'title', 'release_date', 'video_date',
                                'imdb_url'] + [f'genre_{i}' for i in range(19)])
    
    # Re-index to 0-based
    ratings['user_id'] -= 1
    ratings['item_id'] -= 1
    n_users = ratings['user_id'].max() + 1
    n_movies = ratings['item_id'].max() + 1
    
    # User nodes: 0..n_users-1
    # Movie nodes: n_users..n_users+n_movies-1
    src = ratings['user_id'].tolist()
    dst = (ratings['item_id'] + n_users).tolist()
    
    edge_index = torch.tensor([src + dst, dst + src], dtype=torch.long)
    edge_attr = torch.tensor(ratings['rating'].tolist() * 2, dtype=torch.float)
    
    # Movie features: genre one-hot (19 genres)
    genre_cols = [f'genre_{i}' for i in range(19)]
    movie_feat = torch.tensor(movies[genre_cols].values, dtype=torch.float)
    user_feat = torch.ones(n_users, 1)  # dummy feature
    x = torch.cat([
        torch.cat([user_feat, torch.zeros(n_users, 18)], dim=1),
        torch.cat([torch.zeros(n_movies, 1), movie_feat], dim=1),
    ], dim=0)
    
    return Data(x=x, edge_index=edge_index, edge_attr=edge_attr), n_users, n_movies, movies

# data_ml, n_users, n_movies, movies_df = load_movielens('./ml-100k')
# print(f'MovieLens: {n_users} users, {n_movies} movies, {data_ml.num_edges//2} ratings')
'''
print('Uncomment the code above after downloading MovieLens-100K')

---
## Capstone Hints

<details>
<summary>Hint 1 — Negative sampling for bipartite graphs</summary>

```python
# Only sample (user, movie) pairs, not (user, user) or (movie, movie)
neg_edge_index = negative_sampling(
    edge_index=train_pos_edges,
    num_nodes=data.num_nodes,
    num_neg_samples=train_pos_edges.shape[1],
    method='sparse'  # faster for large graphs
)
```
</details>

<details>
<summary>Hint 2 — Hits@K implementation</summary>

```python
@torch.no_grad()
def hits_at_k(z, user_nodes, pos_movie_nodes, all_movie_nodes, k=10):
    hits = 0
    for u, m_pos in zip(user_nodes, pos_movie_nodes):
        # Score user u against all movies
        scores = (z[u] * z[all_movie_nodes]).sum(dim=-1)
        # Get top-K movie indices
        top_k = all_movie_nodes[scores.topk(k).indices]
        if m_pos in top_k:
            hits += 1
    return hits / len(user_nodes)
```
</details>

<details>
<summary>Hint 3 — Heterogeneous graph (advanced)</summary>

```python
from torch_geometric.data import HeteroData
from torch_geometric.nn import to_hetero, SAGEConv

hdata = HeteroData()
hdata['user'].x = user_features
hdata['movie'].x = movie_features
hdata['user', 'rates', 'movie'].edge_index = user_movie_edge_index

# to_hetero converts a homogeneous GNN to handle multiple node/edge types
model_hetero = to_hetero(model, hdata.metadata(), aggr='sum')
```
</details>

---
## What You've Accomplished

You've now built a complete GNN toolkit from scratch:

| Notebook | Topic | Key Skills |
|----------|-------|----------|
| 1 | Graph Foundations | Graphs in NumPy/NetworkX/PyG |
| 2 | Message Passing & GCN | GCN from scratch, PyG MessagePassing |
| 3 | Node Classification | Full training pipeline, overfitting, t-SNE |
| 4 | GAT | Attention mechanisms, visualization |
| 5 | GraphSAGE | Inductive learning, mini-batch training |
| 6 | GIN | Graph classification, pooling, WL test |
| 7 | Benchmarks & Capstone | OGB, link prediction, recommendation system |

**Where to go next:**
- **Heterogeneous GNNs**: `HeteroData`, `HGT`, `HAN`
- **Temporal GNNs**: `TGNN`, dynamic graphs
- **Graph Transformers**: GPS, Graphormer (attention over all pairs)
- **Scalability**: ClusterGCN, GraphSAINT, GAS
- **Molecular GNNs**: MPNN, DimeNet, SchNet
- **OGB Leaderboards**: try to rank on any task!